# V2: Model Creation
## 4.1 Loading Files

In [1]:
import pandas as pd

train_v2_numericalized = pd.read_parquet('data/processed/train_v2_numericalized.parquet')
validation_v2_numericalized = pd.read_parquet('data/processed/validation_v2_numericalized.parquet')

print(f'Train Shape: {train_v2_numericalized.shape}')
print(f'Validation Shape:{validation_v2_numericalized.shape}')

assert 'tokens_ids' in train_v2_numericalized.columns
assert 'tokens_ids' in validation_v2_numericalized

for index in [1, 10, 200]:
    sequence = train_v2_numericalized['tokens_ids'].iloc[index]
    print(f'Sequence of tokens turned into ids: {sequence}')
    print(f'Number of tokens: {len(sequence)}\n')

print(f'Train Columns: {train_v2_numericalized.columns.tolist()}')
print(f'Validation Columns: {validation_v2_numericalized.columns.tolist()}')

Train Shape: (22809, 4)
Validation Shape:(5703, 4)
Sequence of tokens turned into ids: [248 249 337 161 388 258 662 154 592 161 588 154 155 155 247 249 337 161
 388 258 662 154 592 161 588 154 155 164 850 154 155 155]
Number of tokens: 32

Sequence of tokens turned into ids: [247 249 525 154 155]
Number of tokens: 5

Sequence of tokens turned into ids: [247 249 760 154 367 155]
Number of tokens: 6

Train Columns: ['diff', 'top_level_label', 'tokens', 'tokens_ids']
Validation Columns: ['diff', 'top_level_label', 'tokens', 'tokens_ids']


## 4.2 Label to ID

In [2]:
train_unique_labels = train_v2_numericalized['top_level_label'].unique()
validation_unique_labels = validation_v2_numericalized['top_level_label'].unique()
assert not set(validation_unique_labels).difference(set(train_unique_labels))

label_to_ids = {label: label_id for label_id, label in enumerate(sorted(train_unique_labels))}

In [3]:
train_v2_numericalized['label_id'] = train_v2_numericalized['top_level_label'].map(label_to_ids)
assert not train_v2_numericalized['label_id'].isnull().any()

validation_v2_numericalized['label_id'] = validation_v2_numericalized['top_level_label'].map(label_to_ids)
assert not validation_v2_numericalized['label_id'].isnull().any()


for index in [1, 20, 1000]:
    label = train_v2_numericalized['top_level_label'].loc[index]
    label_id = train_v2_numericalized['label_id'].loc[index]
    print(f'Label: {label}\nLabel ID: {label_id}')

Label: call
Label ID: 1
Label: control_flow
Label ID: 2
Label: expression
Label ID: 3


## 4.3 Saving the Label to Id

In [4]:
import json
train_v2_numericalized.to_parquet('data/processed/train_v2_label_id.parquet', index=False)
validation_v2_numericalized.to_parquet('data/processed/validation_v2_label_id.parquet', index=False)

ids_to_labels = {ident: label for label, ident in label_to_ids.items()}
print(ids_to_labels)

with open('data/processed/label_to_id.json', 'w') as file:
    json.dump(label_to_ids, file)

with open('data/processed/id_to_label.json', 'w') as file:
    json.dump(ids_to_labels, file)


{0: 'assignment', 1: 'call', 2: 'control_flow', 3: 'expression', 4: 'identifier'}


## 4.4 Creating the Architecture for the Dataset

In [ ]:
import torch
from torch.utils.data import Dataset

class BugFixDataset(Dataset):
    def __init__(self, dataframe):
        self.dataframe = dataframe
    def __len__(self):
        return len(self.dataframe)
    def __getitem__(self, index):
        tokens_in_ids = self.dataframe['tokens_ids'].iloc[index]
        label_id = self.dataframe['label_id'].iloc[index]
        return torch.tensor(tokens_in_ids, dtype=torch.long), torch.tensor(label_id, dtype=torch.long)

train_dataset = BugFixDataset(train_v2_numericalized)
validation_dataset = BugFixDataset(validation_v2_numericalized)

In [8]:
tokens, label = train_dataset[0]
print(tokens)
print(tokens.dtype, tokens.shape)
print(label)
print(label.dtype, label.shape)

tensor([248, 249, 542, 576, 161, 603, 583, 521, 154, 602, 164, 745, 154, 771,
        161, 211, 155, 155, 242, 247, 249, 542, 576, 161, 603, 583, 521, 154,
        602, 164, 745, 154, 771, 155, 155, 242])
torch.int64 torch.Size([36])
tensor(1)
torch.int64 torch.Size([])


In [17]:
for index in [0, 10, 200]:
    train_tokens, train_label_id = train_dataset[index]
    assert train_tokens.numel() == len(train_v2_numericalized['tokens_ids'].iloc[index])
    assert train_label_id.item() == train_v2_numericalized['label_id'].iloc[index]

    validation_tokens, validation_label_id = validation_dataset[index]
    assert validation_tokens.numel() == len(validation_v2_numericalized['tokens_ids'].iloc[index])
    assert validation_label_id.item() == validation_v2_numericalized['label_id'].iloc[index]

assert len(train_dataset) == len(train_v2_numericalized)
assert len(validation_dataset) == len(validation_v2_numericalized)
